In [1]:
import pandas as pd

In [2]:
df =pd.read_csv('7role.csv')
df

,job_title,job_skills,cleaned_skills,job_category
0,Senior Machine Learning Engineer,"Machine Learning, Programming, Python, Scala, ...","ai, java, machine learning, python, pytorch, s...",AI / Machine Learning Engineer
1,"Principal Software Engineer, ML Accelerators","C++, Python, PyTorch, TensorFlow, MXNet, CUDA,...","c++, computer vision, deep learning, linux, py...",Backend Developer
2,Lead Data Engineer,"Java, Scala, Python, RDBMS, NoSQL, Redshift, S...","agile, aws, azure, cassandra, google cloud, ha...",Data Engineer
3,Senior Data Engineer,"Data Warehouse (DW), Extract/Transform/Load (E...","ai, apache, data warehouse, etl, excel, hadoop...",Data Engineer
4,"Manager, Cyber Risk & Analysis (Machine Learning)","Machine Learning, Generative AI, Cloudbased se...","ai, cism, machine learning",AI / Machine Learning Engineer
...,...,...,...,...
6753,Data Analyst Staff (Denpasar),​​Memantau dan mengelola beberapa perangkat lu...,excel,Data Analyst
6754,Data Analyst Operational,Pendidikan minimal S1 (Sistem Informasi / Tekn...,excel,Data Analyst
6755,Data Analyst Supervisor,Pendidikan minimal S1 jurusan terkait | Pengal...,excel,Data Analyst
6756,Software Developer,Melakukan pengembangan terhadap program-progra...,"docker, git, javascript, laravel, node.js, php...",Backend Developer


In [3]:
df['job_category'].value_counts()

job_category
Data Engineer                     2486
Data Analyst                      1807
AI / Machine Learning Engineer    1032
Data Scientist                     948
Backend Developer                  311
Fullstack Developer                138
Frontend Developer                  36
Name: count, dtype: int64

# adding 3 roles from another dataset because imbalanced data

In [6]:
# 1. Load dataset Kaggle
new = pd.read_csv('/Users/khashiazahira/Desktop/dicoding/CAPSTONE/dataset/support_dataset/postings.csv')

# 2. Definisikan fungsi untuk memfilter HANYA 3 role Web Dev
def get_webdev_category(title):
    if pd.isna(title):
        return 'Drop'
    t = title.lower()
    
    if any(k in t for k in ['full stack', 'fullstack', 'full-stack']):
        return 'Fullstack Developer'
    
    if any(k in t for k in ['frontend', 'front-end', 'front end', 'ui developer', 'react developer', 'vue developer', 'angular developer']):
        return 'Frontend Developer'
        
    if any(k in t for k in ['backend', 'back-end', 'back end', 'software engineer', 'software developer', 'python developer', 'java developer', 'php developer', 'node developer']):
        return 'Backend Developer'
        
    return 'Drop' # Buang role lain (termasuk Data & AI agar tidak duplikat dengan data utamamu)



In [7]:
new['job_category'] = new['job_title'].apply(get_webdev_category)
df_webdev_baru = new[new['job_category'] != 'Drop'].copy()
df_webdev_baru.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8348 entries, 0 to 9379
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   job_title       8348 non-null   object
 1   company         8348 non-null   object
 2   job_location    8348 non-null   object
 3   job_link        8348 non-null   object
 4   first_seen      8348 non-null   object
 5   search_city     8348 non-null   object
 6   search_country  8348 non-null   object
 7   job level       8348 non-null   object
 8   job_type        8348 non-null   object
 9   job_summary     8344 non-null   object
 10  job_skills      8338 non-null   object
 11  job_category    8348 non-null   object
dtypes: object(12)
memory usage: 847.8+ KB


In [8]:
# 4. Standardisasi format agar sama dengan df_final

# Pastikan kolom cleaned_skills ada (copy dari job_skills untuk sementara, bisa di-preprocess lebih lanjut nanti)
df_webdev_baru['cleaned_skills'] = df_webdev_baru['job_skills'].str.lower()

# Pilih hanya kolom yang dibutuhkan
df_webdev_baru = df_webdev_baru[['job_title', 'job_skills', 'cleaned_skills', 'job_category']]


In [9]:
df_webdev_baru.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8348 entries, 0 to 9379
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   job_title       8348 non-null   object
 1   job_skills      8338 non-null   object
 2   cleaned_skills  8338 non-null   object
 3   job_category    8348 non-null   object
dtypes: object(4)
memory usage: 326.1+ KB


# cleaning newest data

In [10]:
df_webdev_baru.isnull().sum()

job_title          0
job_skills        10
cleaned_skills    10
job_category       0
dtype: int64

In [11]:
df_webdev_baru = df_webdev_baru.dropna(subset=['job_skills', 'cleaned_skills'])

In [12]:
df_webdev_baru.duplicated().sum()

np.int64(15)

In [13]:
df_webdev_baru = df_webdev_baru.drop_duplicates()

In [14]:
df_webdev_baru.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8323 entries, 0 to 9379
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   job_title       8323 non-null   object
 1   job_skills      8323 non-null   object
 2   cleaned_skills  8323 non-null   object
 3   job_category    8323 non-null   object
dtypes: object(4)
memory usage: 325.1+ KB


In [15]:
# cek data distribution

df_webdev_baru['job_category'].value_counts()

job_category
Backend Developer      7362
Fullstack Developer     723
Frontend Developer      238
Name: count, dtype: int64

In [16]:
# potong dataset

# Ambil maksimal 800 baris secara acak untuk tiap profesi
df_ideal = df_webdev_baru.groupby('job_category').apply(
    lambda x: x.sample(n=min(len(x), 800), random_state=42)
).reset_index(drop=True)

print(df_ideal['job_category'].value_counts())

job_category
Backend Developer      800
Fullstack Developer    723
Frontend Developer     238
Name: count, dtype: int64


/var/folders/fh/3rw7ypm52s79sv5l1sp19zg00000gn/T/ipykernel_35724/494872303.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_ideal = df_webdev_baru.groupby('job_category').apply(


# gabungin sama dataset yang sebelumnya

In [17]:
df_gabungan = pd.concat([df, df_ideal], ignore_index=True)

In [18]:
print(df_gabungan['job_category'].value_counts())

job_category
Data Engineer                     2486
Data Analyst                      1807
Backend Developer                 1111
AI / Machine Learning Engineer    1032
Data Scientist                     948
Fullstack Developer                861
Frontend Developer                 274
Name: count, dtype: int64


In [21]:
# dataset lain
# 1. Load dataset Kaggle
df_kaggle = pd.read_csv('/Users/khashiazahira/Desktop/dicoding/CAPSTONE/dataset/support_dataset/job_dataset.csv')

In [22]:
# 3. Aplikasikan filter
df_kaggle['job_category'] = df_kaggle['Title'].apply(get_webdev_category)
df_webdev = df_kaggle[df_kaggle['job_category'] != 'Drop'].copy()

# 4. Standardisasi format agar sama dengan df_final
# Rename kolom agar match
df_webdev = df_webdev.rename(columns={'Title': 'job_title', 'Skills': 'job_skills'})

# Ubah format semicolon (;) menjadi comma (,)
df_webdev['job_skills'] = df_webdev['job_skills'].astype(str).str.replace(';', ', ')

# Pastikan kolom cleaned_skills ada (copy dari job_skills untuk sementara, bisa di-preprocess lebih lanjut nanti)
df_webdev['cleaned_skills'] = df_webdev['job_skills'].str.lower()

# Pilih hanya kolom yang dibutuhkan
df_webdev = df_webdev[['job_title', 'job_skills', 'cleaned_skills', 'job_category']]

# 5. GABUNGKAN dengan data Data & AI milikmu sebelumnya
# Asumsi dataset utamamu yang sudah difilter bernama df_data_ai
df_ultimate_final = pd.concat([df_gabungan, df_webdev], ignore_index=True)



In [23]:
df_ultimate_final.duplicated().sum()

np.int64(13)

In [24]:
df_ultimate_final = df_ultimate_final.drop_duplicates()

In [25]:
df_ultimate_final.duplicated().sum()

np.int64(0)

In [26]:
df_ultimate_final.isnull().sum()

job_title         0
job_skills        0
cleaned_skills    0
job_category      0
dtype: int64

In [27]:
df_ultimate_final = df_ultimate_final.reset_index(drop=True)

df_ultimate_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8644 entries, 0 to 8643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   job_title       8644 non-null   object
 1   job_skills      8644 non-null   object
 2   cleaned_skills  8644 non-null   object
 3   job_category    8644 non-null   object
dtypes: object(4)
memory usage: 270.2+ KB


In [28]:
# 6. Cek distribusi akhir untuk ke-7 role!
print("=== Distribusi 7 Role Capstone Lengkap ===")
print(df_ultimate_final['job_category'].value_counts())

=== Distribusi 7 Role Capstone Lengkap ===
job_category
Data Engineer                     2486
Data Analyst                      1807
Backend Developer                 1206
AI / Machine Learning Engineer    1032
Data Scientist                     948
Fullstack Developer                880
Frontend Developer                 285
Name: count, dtype: int64


In [29]:
df_ultimate_final.to_csv('bismillah_fix_dataset.csv', index=False)